In [6]:
# sweep threshold according to entry-exit rule
# see actionPlan_binary.md
import json
import numpy as np
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.metrics import classification_report, f1_score

def find_project_root(marker=".git"):
    p = Path.cwd()
    while p != p.parent:
        if (p / marker).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project root not found")


# ── Paths ────────────────────────────────────────────────────────────────────

ROOT = find_project_root() 
MODEL_PATH = ROOT / "data/mlData/model/delta/Delta_final.pt"
TEST_PATH  = ROOT / "data/mlData/processed/202603-vX-test-regime-mapped-ohlc.jsonl"

SEQ_LEN    = 20
BATCH_SIZE = 512
LABEL_MAP  = {-1: 0, 0: 1, 1: 2}
CLASS_NAMES = ["DOWN", "NO_HIT", "UP"]

# ── Model architecture (mirrors lstmXII.py TripleBarrierLSTM) ─────────────
class TripleBarrierLSTM(nn.Module):
    def __init__(self, n_features, seq_len=20,
                 lstm1_hidden=128, lstm2_hidden=64, dense_hidden=32,
                 input_dropout=0.10, recurrent_dropout=0.20, dense_dropout=0.20, n_classes=3):
        super().__init__()
        self.input_dropout = nn.Dropout(p=input_dropout)
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=lstm1_hidden,
                            num_layers=2, batch_first=True,
                            dropout=recurrent_dropout, bidirectional=False)
        self.hidden_proj = nn.Linear(lstm1_hidden, lstm2_hidden)
        self.batch_norm  = nn.BatchNorm1d(lstm2_hidden)
        self.dense = nn.Sequential(
            nn.Linear(lstm2_hidden, dense_hidden),
            nn.ReLU(),
            nn.Dropout(p=dense_dropout),
            nn.Linear(dense_hidden, n_classes),
        )
        self.softmax = nn.Softmax(dim=-1)

    def forward(self, x, return_logits=True):
        x = self.input_dropout(x)
        _, (h_n, _) = self.lstm(x)
        ctx = h_n[-1]
        ctx = self.hidden_proj(ctx)
        ctx = self.batch_norm(ctx)
        logits = self.dense(ctx)
        return logits if return_logits else self.softmax(logits)

# ── Load checkpoint ───────────────────────────────────────────────────────
ckpt         = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
feature_cols = ckpt["feature_cols"]
mkw          = ckpt["model_kwargs"]
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TripleBarrierLSTM(n_features=mkw["n_features"], seq_len=mkw["seq_len"])
model.load_state_dict(ckpt["model_state"])
model.to(device).eval()

if 'val_loss' not in ckpt:
    ckpt['val_loss'] = 0
if 'val_acc' not in ckpt:
    ckpt['val_acc'] = 0
if 'val_f1' not in ckpt:
    ckpt['val_f1'] = 0

print(f"Loaded  : {MODEL_PATH.name}")
print(f"Run     : {ckpt['run_name']}  |  best epoch: {ckpt['best_epoch']}")
print(f"Val     : loss={ckpt['val_loss']:.4f}  acc={ckpt['val_acc']:.4f}  F1={ckpt['val_f1']:.4f}")
print(f"Features: {len(feature_cols)}  →  {feature_cols}")
print(f"Device  : {device}")

# ── Load test data ────────────────────────────────────────────────────────
X_rows, y_rows = [], []
with open(TEST_PATH) as f:
    for line in f:
        row = json.loads(line)
        lbl = row.get("label")
        if lbl is None:
            continue
        X_rows.append([row[c] for c in feature_cols])
        y_rows.append(LABEL_MAP[int(lbl)])

X = np.array(X_rows, dtype=np.float32)
y = np.array(y_rows, dtype=np.int64)
print(f"\nTest    : {len(X):,} rows  →  {len(X) - SEQ_LEN:,} sequences")

# ── Sliding-window inference ──────────────────────────────────────────────
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.long)

all_probs, all_true = [], []
with torch.no_grad():
    for start in range(0, len(X_t) - SEQ_LEN, BATCH_SIZE):
        end     = min(start + BATCH_SIZE, len(X_t) - SEQ_LEN)
        seqs    = torch.stack([X_t[i:i + SEQ_LEN] for i in range(start, end)]).to(device)
        labels  = y_t[start + SEQ_LEN : end + SEQ_LEN]
        probs   = model(seqs, return_logits=False).cpu().numpy()
        all_probs.append(probs)
        all_true.extend(labels.tolist())

probs_arr  = np.vstack(all_probs)           # [N, 3]
y_pred_arr = probs_arr.argmax(axis=1)
y_true_arr = np.array(all_true)

# ── Classification report ─────────────────────────────────────────────────
test_f1  = f1_score(y_true_arr, y_pred_arr, average="macro", zero_division=0)
val_f1   = ckpt["val_f1"]
gap      = val_f1 - test_f1

print(f"\n{'='*60}")
print(f"  Val  F1 macro : {val_f1:.4f}")
print(f"  Test F1 macro : {test_f1:.4f}")
print(f"  Regime gap    : {gap:.4f}", end="  ")
if   gap < 0.02:  print("← generalises well")
elif gap < 0.08:  print("← expected structural gap (normal)")
elif gap < 0.15:  print("← WARN: moderate overfitting to bull regime")
else:             print("← ALARM: severe regime overfitting")
print(f"{'='*60}")
print(classification_report(y_true_arr, y_pred_arr, target_names=CLASS_NAMES, digits=4))


Loaded  : Delta_final.pt
Run     : Run4d_AFL_120  |  best epoch: 9
Val     : loss=0.0000  acc=0.0000  F1=0.0000
Features: 22  →  ['ROC_3', 'ROC_5', 'ROC_10', 'MOM_3', 'RETURNS_1', 'ATR_5', 'ATR_14', 'ATR_RATIO', 'ATR_NORM_ROC', 'RANGE_RATIO', 'RSI_14', 'RSI_SLOPE', 'STOCH_K', 'CCI_5', 'DELTA_1', 'DELTA_3', 'VOL_SPIKE', 'DIST_HIGH_5', 'DIST_LOW_5', 'DIST_HIGH_10', 'DIST_LOW_10', 'RANGE_POS']
Device  : cpu

Test    : 54,586 rows  →  54,566 sequences

  Val  F1 macro : 0.0000
  Test F1 macro : 0.2713
  Regime gap    : -0.2713  ← generalises well
              precision    recall  f1-score   support

        DOWN     0.3356    0.5246    0.4093     18314
      NO_HIT     0.0000    0.0000    0.0000     18133
          UP     0.3437    0.4919    0.4046     18119

    accuracy                         0.3394     54566
   macro avg     0.2264    0.3388    0.2713     54566
weighted avg     0.2267    0.3394    0.2717     54566



/Users/aimliu/.pyenv/versions/3.14.0/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/aimliu/.pyenv/versions/3.14.0/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/aimliu/.pyenv/versions/3.14.0/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{me

In [9]:
# ── Threshold sweep × regime  (follows actionPlan_binary.md) ─────────────────
# probs_arr, SEQ_LEN, TEST_PATH already in scope from cell 1
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ── Constants ────────────────────────────────────────────────────────────────
HORIZONS    = 3
THRESHOLDS  = np.round(np.arange(0.01, 0.61, 0.01), 2)  # 0.25 → 0.75 step 0.05
MIN_SIGNALS = 30

IDX_DOWN, IDX_NONE, IDX_UP = 0, 1, 2

# ── Reload aligned rows (trim first SEQ_LEN rows — no prediction for them) ──
raw_rows = [json.loads(l) for l in Path(TEST_PATH).read_text().splitlines()]
rows     = raw_rows[SEQ_LEN:]          # aligns with probs_arr index-for-index

assert len(rows) == len(probs_arr), (
    f"Row count mismatch: rows={len(rows)}, probs_arr={len(probs_arr)}"
)

test_timestamps = pd.to_datetime([r["timestamp"] for r in rows], unit="ms", utc=True)
entry_open      = np.array([r["entry_open"]  for r in rows], dtype=np.float64)
exit_close      = np.array([r["exit_close"]  for r in rows], dtype=np.float64)
regime_labels   = np.array([r["regime"]      for r in rows])

ALL_REGIMES = sorted(set(regime_labels))
print(f"Rows: {len(rows):,}  |  Regimes: {ALL_REGIMES}")
print(f"Threshold sweep: {THRESHOLDS}")

# ── Helpers ──────────────────────────────────────────────────────────────────
def apply_one_position_filter(signal_indices, timestamps, exit_bars=3):
    """
    1. Entry bar (idx+1) must open on HH:00/15/30/45 boundary.
    2. One-position rule: next signal starts only after current trade exits.
    """
    taken     = []
    last_exit = -1
    n         = len(timestamps)
    for idx in signal_indices:
        entry_idx = idx + 1
        if entry_idx >= n:
            continue
        if timestamps[entry_idx].minute % 15 != 0:
            continue
        if idx > last_exit:
            taken.append(idx)
            last_exit = idx + exit_bars
    return np.array(taken, dtype=np.int64)


def max_drawdown(outcomes):
    """Peak-to-trough drawdown of cumulative +1/-1 sequence."""
    cum  = np.cumsum(outcomes)
    peak = np.maximum.accumulate(cum)
    return int((peak - cum).max())


# ── Threshold sweep ───────────────────────────────────────────────────────────
results = []

for thr in THRESHOLDS:
    long_mask  = probs_arr[:, IDX_UP]   > thr
    short_mask = probs_arr[:, IDX_DOWN] > thr

    long_idx  = np.where(long_mask)[0]
    short_idx = np.where(short_mask)[0]

    all_idx = np.concatenate([long_idx, short_idx])
    all_dir = np.concatenate([np.ones(len(long_idx)), -np.ones(len(short_idx))])
    order   = np.argsort(all_idx)
    all_idx, all_dir = all_idx[order], all_dir[order]

    # One-position filter + 15-min boundary gate
    taken_arr  = apply_one_position_filter(all_idx, test_timestamps, exit_bars=HORIZONS)
    taken_set  = set(taken_arr.tolist())
    keep       = np.array([i for i, idx in enumerate(all_idx) if idx in taken_set])

    if len(keep) == 0:
        continue

    idx_taken = all_idx[keep]
    dir_taken = all_dir[keep]

    # Drop signals too close to end (no exit bar available)
    valid     = (idx_taken + HORIZONS) < len(probs_arr)
    idx_taken = idx_taken[valid]
    dir_taken = dir_taken[valid]

    rgm_taken  = regime_labels[idx_taken]
    price_up   = exit_close[idx_taken] > entry_open[idx_taken]
    price_down = exit_close[idx_taken] < entry_open[idx_taken]
    correct    = np.where(dir_taken == 1, price_up, price_down)

    for direction, dir_label in [(1, "LONG"), (-1, "SHORT")]:
        for rgm in ALL_REGIMES:
            mask = (dir_taken == direction) & (rgm_taken == rgm)
            n    = mask.sum()
            if n < MIN_SIGNALS:
                continue
            c    = correct[mask]
            wr   = c.mean()
            edge = 2 * wr - 1
            mdd  = max_drawdown(np.where(c, 1, -1))
            results.append({
                "threshold": thr,
                "direction": dir_label,
                "regime":    rgm,
                "n":         int(n),
                "win_rate":  round(float(wr),   4),
                "edge":      round(float(edge), 4),
                "max_dd":    mdd,
            })

results_df = pd.DataFrame(results)
print(f"\nResult rows: {len(results_df)}")
results_df.head()


Rows: 54,566  |  Regimes: [np.str_('ACCUMULATION'), np.str_('BEARISH'), np.str_('BULLISH'), np.str_('CORRECTION'), np.str_('DISTRIBUTION'), np.str_('RECOVERY')]
Threshold sweep: [0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1  0.11 0.12 0.13 0.14
 0.15 0.16 0.17 0.18 0.19 0.2  0.21 0.22 0.23 0.24 0.25 0.26 0.27 0.28
 0.29 0.3  0.31 0.32 0.33 0.34 0.35 0.36 0.37 0.38 0.39 0.4  0.41 0.42
 0.43 0.44 0.45 0.46 0.47 0.48 0.49 0.5  0.51 0.52 0.53 0.54 0.55 0.56
 0.57 0.58 0.59 0.6 ]

Result rows: 606


,threshold,direction,regime,n,win_rate,edge,max_dd
0,0.01,LONG,ACCUMULATION,2201,0.5139,0.0277,39
1,0.01,LONG,BEARISH,3593,0.5010,0.0019,85
2,0.01,LONG,BULLISH,1393,0.4939,-0.0122,49
3,0.01,LONG,CORRECTION,542,0.5148,0.0295,15
4,0.01,LONG,DISTRIBUTION,1197,0.4946,-0.0109,41


In [10]:
# ── Report: edge tables per direction × regime ───────────────────────────────

# Regime distribution summary
regime_counts = pd.Series(regime_labels).value_counts().sort_index()
total         = regime_counts.sum()
print("Regime distribution (full test set, post SEQ_LEN trim)")
print(f"{'='*45}")
for rgm, cnt in regime_counts.items():
    print(f"  {rgm:<14} {cnt:>6,}   ({cnt/total*100:5.1f}%)")
print(f"  {'TOTAL':<14} {total:>6,}   (100.0%)")

for direction in ["LONG", "SHORT"]:
    for rgm in ALL_REGIMES:
        sub = results_df[
            (results_df.direction == direction) &
            (results_df.regime    == rgm)
        ].sort_values("threshold")

        if sub.empty:
            continue

        rgm_total = regime_counts.get(rgm, 0)
        rgm_pct   = rgm_total / total * 100

        best_row = sub.loc[sub["edge"].idxmax()]
        print(f"\n{'='*72}")
        print(f"  {direction} | {rgm}   [{rgm_total:,} bars · {rgm_pct:.1f}% of test set]")
        print(f"{'='*72}")
        print(sub.to_string(index=False))
        print(f"\n  ► Best threshold: {best_row.threshold}  "
              f"edge={best_row.edge:.4f}  n={int(best_row.n)}  "
              f"win_rate={best_row.win_rate:.3f}  max_dd={int(best_row.max_dd)}")


Regime distribution (full test set, post SEQ_LEN trim)
  ACCUMULATION   13,204   ( 24.2%)
  BEARISH        21,567   ( 39.5%)
  BULLISH         8,337   ( 15.3%)
  CORRECTION      3,237   (  5.9%)
  DISTRIBUTION    7,192   ( 13.2%)
  RECOVERY        1,029   (  1.9%)
  TOTAL          54,566   (100.0%)

  LONG | ACCUMULATION   [13,204 bars · 24.2% of test set]
 threshold direction       regime    n  win_rate   edge  max_dd
      0.01      LONG ACCUMULATION 2201    0.5139 0.0277      39
      0.02      LONG ACCUMULATION 2201    0.5139 0.0277      39
      0.03      LONG ACCUMULATION 2201    0.5139 0.0277      39
      0.04      LONG ACCUMULATION 2201    0.5139 0.0277      39
      0.05      LONG ACCUMULATION 2201    0.5139 0.0277      39
      0.06      LONG ACCUMULATION 2201    0.5139 0.0277      39
      0.07      LONG ACCUMULATION 2201    0.5139 0.0277      39
      0.08      LONG ACCUMULATION 2201    0.5139 0.0277      39
      0.09      LONG ACCUMULATION 2201    0.5139 0.0277      39
 